In [3]:
import os
import string
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# Prepare splits

In [2]:
SPLITS_PATH_TO_SAVE = "data/C61_splits.csv"
FILES = [
    'C61_matched_patients.csv',
]
df = pd.DataFrame()
for file in FILES:
    df = pd.concat([df, pd.read_csv(file, sep=',')], ignore_index=True)
df.shape

(782, 10)

In [4]:
df["icd10_category"].value_counts()

icd10_category
C61    391
Name: count, dtype: int64

In [6]:
splits = train_test_split(df, test_size = 0.2, random_state=42, shuffle=True, stratify=df["icd10_category"].fillna("None"))
splits[0]["group"] = "train"
splits[1]["group"] = "test"
splits = pd.concat(splits)

In [11]:
if not os.path.exists(os.path.dirname(SPLITS_PATH_TO_SAVE)):
    os.makedirs(os.path.dirname(SPLITS_PATH_TO_SAVE))
splits.to_csv(SPLITS_PATH_TO_SAVE, index=False)

# Merge train-test split with features

In [4]:
DIR = "./"
NOSOLOGY = "C61"
SPLITS_PATH = f"{DIR}{NOSOLOGY}_splits.csv"
SUFFIXES = [
    # "deepseek-ai_DeepSeek-V3_features_max",
    # "Qwen_Qwen3-235B-A22B-Instruct-2507_features_max",
    # "yandex_YandexGPT-5-Lite-8B-instruct_features_max",
    "baseline_features"
]

for SUFFIX in tqdm(SUFFIXES):
    FEATURES_PATH = f"{DIR}{NOSOLOGY}_{SUFFIX}.csv"
    DATASET_PATH = f"{DIR}{NOSOLOGY}_{SUFFIX}.csv"

    splits = pd.read_csv(SPLITS_PATH)
    features = pd.read_csv(FEATURES_PATH)

    df = pd.merge(
        features,
        splits[["subject_id", "group"]],
        on="subject_id"
    )
    df.to_csv(DATASET_PATH, index=False)

100%|██████████| 1/1 [00:00<00:00,  3.81it/s]
